# YOLOv8n-seg — pipe-crack-detection v1i (SAM2)

런타임 → 런타임 유형 변경 → T4 GPU

`pipe-crack-detection.v1i.sam2.zip`(142 MB)을 아래 셀에서 업로드한다. SAM2 export는 이미지마다 RLE 마스크 JSON이 붙은 평탄한 구조라 YOLO segmentation 라벨로 변환한 뒤 학습한다. 클래스는 `Crack` 1종.

In [ ]:
%pip install -q ultralytics pycocotools

import torch
assert torch.cuda.is_available(), "GPU 런타임 아님"
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import files

RUNS_DIR = Path('/content/runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ZIP = 'pipe-crack-detection.v1i.sam2.zip'

ZIP_PATH = next((p for p in (Path.cwd() / DATASET_ZIP, Path('/content') / DATASET_ZIP)
                 if p.is_file()), None)

if ZIP_PATH is None:
    print(f"{DATASET_ZIP} 선택")
    uploaded = files.upload()
    ZIP_PATH = Path('/content') / next(iter(uploaded))

print(ZIP_PATH, f"{ZIP_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
import shutil, json, yaml
import numpy as np
import cv2
from pycocotools import mask as maskutil

RAW_DIR = Path('/content/dataset_raw')
DATASET_DIR = Path('/content/dataset')
CLASS_NAMES = ['Crack']
CRACK_IDX = 0

for d in (RAW_DIR, DATASET_DIR):
    if d.exists():
        shutil.rmtree(d)
RAW_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), str(RAW_DIR))


def rle_to_polygons(seg):
    rle = dict(seg)
    if isinstance(rle['counts'], str):
        rle['counts'] = rle['counts'].encode()
    m = maskutil.decode(rle)
    h, w = m.shape
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in contours:
        if cv2.contourArea(c) < 4:
            continue
        p = cv2.approxPolyDP(c, 0.002 * cv2.arcLength(c, True), True).reshape(-1, 2).astype(float)
        if len(p) < 3:
            continue
        p[:, 0] /= w
        p[:, 1] /= h
        out.append(np.clip(p, 0.0, 1.0))
    return out


stats = {}
for split in ('train', 'valid', 'test'):
    src = RAW_DIR / split
    if not src.is_dir():
        continue
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)

    n_img = n_ann = n_drop = 0
    for jf in sorted(src.glob('*.json')):
        d = json.loads(jf.read_text())
        image = src / d['image']['file_name']
        if not image.is_file():
            continue
        lines = []
        for a in d['annotations']:
            polys = rle_to_polygons(a['segmentation'])
            if not polys:
                n_drop += 1
                continue
            for p in polys:
                lines.append(f"{CRACK_IDX} " + ' '.join(f"{v:.6f}" for v in p.reshape(-1)))
                n_ann += 1
        shutil.copy(image, img_dir / image.name)
        (lbl_dir / f"{image.stem}.txt").write_text('\n'.join(lines))
        n_img += 1
    stats[split] = (n_img, n_ann, n_drop)

DATA_YAML = DATASET_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'train': str(DATASET_DIR / 'train' / 'images'),
    'val': str(DATASET_DIR / 'valid' / 'images'),
    'test': str(DATASET_DIR / 'test' / 'images'),
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}, allow_unicode=True, sort_keys=False))

for k, (i, a, dr) in stats.items():
    print(f"{k:6s} 이미지 {i:5d}  폴리곤 {a:5d}  버림 {dr}")
print(DATA_YAML.read_text())

In [ ]:
TAG = 'yolov8n-seg'
WEIGHTS = 'yolov8n-seg.pt'

QUICK_MODE = True

TRAIN_ARGS = dict(
    data=str(DATA_YAML),
    epochs=25 if QUICK_MODE else 60,
    patience=10 if QUICK_MODE else 20,
    imgsz=640,
    batch=16,
    seed=0,
    mosaic=0.5,
    close_mosaic=10,
    project=str(RUNS_DIR),
    exist_ok=True,
    plots=True,
    verbose=False,
)

TARGET_HZ = 10.0
LATENCY_BUDGET_MS = 1000.0 / TARGET_HZ

print(TAG, '| epochs', TRAIN_ARGS['epochs'], '| budget', LATENCY_BUDGET_MS, 'ms')

In [ ]:
import time, gc
from ultralytics import YOLO

started = time.time()
model = YOLO(WEIGHTS)
result = model.train(name=TAG, **TRAIN_ARGS)
TRAIN_SEC = time.time() - started

SAVE_DIR = Path(result.save_dir)
BEST = SAVE_DIR / 'weights' / 'best.pt'
print(f"{TRAIN_SEC / 60:.1f}분  {BEST}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import pandas as pd

test_images = [str(p) for p in sorted((DATASET_DIR / 'test' / 'images').glob('*.jpg'))]
LATENCY_SAMPLES = test_images[:60]

model = YOLO(str(BEST))

try:
    _, n_params, _, gflops = model.info(verbose=False)
except Exception:
    n_params, gflops = float('nan'), float('nan')

m = model.val(data=str(DATA_YAML), split='test', imgsz=640, verbose=False)

model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
preds = model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
latency = float(np.mean([sum(p.speed.values()) for p in preds]))

conf = np.concatenate([p.boxes.conf.cpu().numpy() for p in preds if len(p.boxes)]) \
    if any(len(p.boxes) for p in preds) else np.array([0.0])

df = pd.DataFrame([{
    '모델': TAG,
    'Mask mAP50': float(m.seg.map50),
    'Mask mAP50-95': float(m.seg.map),
    'Box mAP50': float(m.box.map50),
    'Box mAP50-95': float(m.box.map),
    '파라미터(M)': n_params / 1e6,
    'GFLOPs': gflops,
    '지연(ms)': latency,
    '10Hz': latency <= LATENCY_BUDGET_MS,
    '최대신뢰도': float(conf.max()),
    '신뢰도0.8초과비율': float((conf > 0.8).mean()),
    '학습(분)': TRAIN_SEC / 60,
}])

del model
gc.collect()
torch.cuda.empty_cache()

df

In [ ]:
from IPython.display import Image as IPyImage, display

for name in ('results.png', 'MaskPR_curve.png', 'BoxPR_curve.png', 'confusion_matrix_normalized.png'):
    p = SAVE_DIR / name
    if p.exists():
        print(name)
        display(IPyImage(filename=str(p), width=900))

In [ ]:
import matplotlib.pyplot as plt

model = YOLO(str(BEST))
preds = model.predict(source=test_images[:6], imgsz=640, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, pred in zip(axes.ravel(), preds):
    ax.imshow(pred.plot()[:, :, ::-1])
    ax.axis('off')
plt.tight_layout()
plt.show()

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
OUT = Path('/content/yolov8n_seg_out')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

df.to_csv(OUT / 'yolov8n_seg_result.csv', index=False)
shutil.copy(BEST, OUT / 'yolov8n_seg_best.pt')
for f in ('results.png', 'results.csv', 'MaskPR_curve.png', 'BoxPR_curve.png',
          'confusion_matrix_normalized.png'):
    if (SAVE_DIR / f).exists():
        shutil.copy(SAVE_DIR / f, OUT / f'yolov8n_seg_{f}')

archive = shutil.make_archive('/content/yolov8n_seg_out', 'zip', str(OUT))
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
print(sorted(p.name for p in OUT.iterdir()))

files.download(archive)